<a href="https://colab.research.google.com/github/navya039/supernan-ai-dubbing/blob/main/notebooks/colab_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Sat Mar  7 08:27:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Installing core tools needed for video processing
!pip install gdown -q
!apt-get install -y ffmpeg -q

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


In [3]:
!ffmpeg -version


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-l

In [4]:
import os

!gdown "1uDzLVEow_gAJsXnNjbSoskzVLZ4d7opW" -O supernan_video.mp4

# Verify download
size = os.path.getsize("supernan_video.mp4") / 1024 / 1024
print(f"✅ Video downloaded: {size:.1f} MB")

Downloading...
From (original): https://drive.google.com/uc?id=1uDzLVEow_gAJsXnNjbSoskzVLZ4d7opW
From (redirected): https://drive.google.com/uc?id=1uDzLVEow_gAJsXnNjbSoskzVLZ4d7opW&confirm=t&uuid=e3a08381-ff28-452e-ad17-5f410ded3e60
To: /content/supernan_video.mp4
100% 622M/622M [00:13<00:00, 44.6MB/s]
✅ Video downloaded: 593.1 MB


In [5]:
# Extracting the 0:15 to 0:30 segment from the full video
# -ss = start time, -to = end time, -c:v copy = no re-encoding (fast)
!ffmpeg -i supernan_video.mp4 -ss 00:00:15 -to 00:00:30 -c:v libx264 -c:a aac clip_15s.mp4

# Extract just the audio from the clip (needed for Whisper + voice reference)
!ffmpeg -i clip_15s.mp4 -vn -acodec pcm_s16le -ar 16000 -ac 1 clip_audio.wav

# Verify both files created
clip_size = os.path.getsize("clip_15s.mp4") / 1024 / 1024
audio_size = os.path.getsize("clip_audio.wav") / 1024 / 1024
print(f" Video clip: {clip_size:.1f} MB")
print(f" Audio extracted: {audio_size:.1f} MB")

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [6]:
# Installing Whisper - using large-v3 for best accuracy
!pip install openai-whisper -q
!pip install torch -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 29.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [7]:
import whisper

# Loading large-v3 - most accurate model, worth the extra load time
model = whisper.load_model("large-v3")

# Transcribe with word-level timestamps - this is key for chunk alignment later
result = model.transcribe(
    "clip_audio.wav",
    language="en",
    word_timestamps=True,
    verbose=False
)

# Print full transcript
print("📝 Full Transcript:")
print(result["text"])

print("\n⏱️ Word-level timestamps:")
for segment in result["segments"]:
    for word in segment["words"]:
        print(f"  {word['start']:.2f}s → {word['end']:.2f}s : {word['word']}")

100%|██████████████████████████████████████| 2.88G/2.88G [00:21<00:00, 147MiB/s]
100%|██████████| 1501/1501 [00:09<00:00, 152.14frames/s]

📝 Full Transcript:
 So now let's see how to protect your hygiene First step is to brush your teeth before booking

⏱️ Word-level timestamps:
  0.00s → 1.12s :  So
  1.12s → 1.32s :  now
  1.32s → 1.74s :  let's
  1.74s → 1.88s :  see
  1.88s → 2.02s :  how
  2.02s → 2.98s :  to
  2.98s → 3.20s :  protect
  3.20s → 3.58s :  your
  3.58s → 3.58s :  hygiene
  3.58s → 5.00s :  First
  5.00s → 6.22s :  step
  6.22s → 6.40s :  is
  6.40s → 6.90s :  to
  6.90s → 9.48s :  brush
  9.48s → 9.74s :  your
  9.74s → 10.22s :  teeth
  10.22s → 14.36s :  before
  14.36s → 14.98s :  booking


In [8]:
import json

# Saving full segment data - we'll need timestamps during TTS alignment
transcript_data = {
    "full_text": result["text"],
    "segments": result["segments"],
    "language": result["language"]
}

with open("transcript.json", "w") as f:
    json.dump(transcript_data, f, indent=2)

# Also save clean text with manual fix applied
clean_text = result["text"].replace("booking", "looking")
with open("transcript_clean.txt", "w") as f:
    f.write(clean_text)

print("✅ Transcript saved!")
print(f"📝 Clean text: {clean_text}")

✅ Transcript saved!
📝 Clean text:  So now let's see how to protect your hygiene First step is to brush your teeth before looking
